# 5.2 · Heaps and priority queues

*Trees* · 10 exercises

Always reach the most urgent item in O(log N) with a heap stored in an array.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ch-5.2) · [Answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ch-5.2) · [Concept guide](https://bloomcode.github.io/dsa/guide/#ch-5.2)

**Save your work:** this notebook is read-only, so first use **File → Save a copy in Drive** and work in that copy.

**How to work:** run the setup cell once. For each exercise, replace the stub in its code cell, run the cell, then run the test cell under it. Later exercises often reuse earlier ones, so go in order. Stuck? Each exercise links to its tested solution in the answer key.

In [ ]:
# Setup: run once.
import heapq
import itertools
import random

try:
    import pytest
except ImportError:  # the tests only need pytest.raises; a small stand-in keeps the notebook self-contained
    import re as _re

    class _Raises:
        def __init__(self, kind, match=None):
            self.kind = kind
            self.match = match
            self.value = None

        def __enter__(self):
            return self

        def __exit__(self, kind, value, tb):
            if kind is None:
                raise AssertionError(f"did not raise {self.kind.__name__}")
            if not issubclass(kind, self.kind):
                return False
            self.value = value
            if self.match is not None and not _re.search(self.match, str(value)):
                raise AssertionError(f"exception message {str(value)!r} did not match {self.match!r}")
            return True

    class pytest:  # noqa: N801
        raises = _Raises

def _check_cmp(l, r, op, ls, rs):
    __tracebackhide__ = True
    passed = False
    if op == "==": passed = (l == r)
    elif op == "!=": passed = (l != r)
    elif op == "<": passed = (l < r)
    elif op == "<=": passed = (l <= r)
    elif op == ">": passed = (l > r)
    elif op == ">=": passed = (l >= r)
    elif op == "is": passed = (l is r)
    elif op == "is not": passed = (l is not r)
    elif op == "in": passed = (l in r)
    elif op == "not in": passed = (l not in r)
    if not passed:
        import inspect
        caller = inspect.currentframe().f_back
        locs = {k: v for k, v in caller.f_locals.items()
                if not k.startswith("_") and type(v) in (int, float, str, bool, list, tuple, dict, set)}
        lines = [
            f"Assertion Failed: {ls} {op} {rs}",
            f"  Expected: {rs} -> {r!r} ({type(r).__name__})",
            f"  Got:      {ls} -> {l!r} ({type(l).__name__})"
        ]
        if locs:
            lines.append("  Context:  " + ", ".join(f"{k} = {v!r}" for k, v in locs.items()))
        if isinstance(l, (list, tuple, str)) and isinstance(r, (list, tuple, str)):
            if len(l) != len(r):
                lines.append(f"  Length mismatch: got {len(l)} items, expected {len(r)}")
            if isinstance(l, (list, tuple)) and isinstance(r, (list, tuple)):
                for i, (a, b) in enumerate(zip(l, r)):
                    if a != b:
                        lines.append(f"  First diff at index {i}: got {a!r}, expected {b!r}")
                        break
        raise AssertionError(chr(10) + chr(10).join(lines))

def _check_expr(val, expr):
    __tracebackhide__ = True
    if not val:
        raise AssertionError(chr(10) + f"Assertion Failed: expected {expr} to be truthy, got {val!r}")

try:
    _ip = get_ipython()
    if _ip is not None:
        import ast as _ast

        class _AssertRewriter(_ast.NodeTransformer):
            def __init__(self):
                self._tid = 0

            def visit_Assert(self, node):
                test = node.test
                if isinstance(test, _ast.BoolOp) and isinstance(test.op, _ast.And):
                    res = []
                    for sub in test.values:
                        res.extend(self.visit_Assert(_ast.Assert(test=sub, msg=node.msg)))
                    return res
                if isinstance(test, _ast.Compare) and len(test.ops) == 1:
                    op_map = {_ast.Eq: "==", _ast.NotEq: "!=", _ast.Lt: "<", _ast.LtE: "<=",
                              _ast.Gt: ">", _ast.GtE: ">=", _ast.Is: "is", _ast.IsNot: "is not",
                              _ast.In: "in", _ast.NotIn: "not in"}
                    op_cls = type(test.ops[0])
                    if op_cls in op_map:
                        self._tid += 1
                        la, ra = f"_a_{self._tid}", f"_b_{self._tid}"
                        return [
                            _ast.Assign(targets=[_ast.Name(id=la, ctx=_ast.Store())], value=test.left),
                            _ast.Assign(targets=[_ast.Name(id=ra, ctx=_ast.Store())], value=test.comparators[0]),
                            _ast.Expr(value=_ast.Call(
                                func=_ast.Name(id="_check_cmp", ctx=_ast.Load()),
                                args=[_ast.Name(id=la, ctx=_ast.Load()), _ast.Name(id=ra, ctx=_ast.Load()),
                                      _ast.Constant(value=op_map[op_cls]), _ast.Constant(value=_ast.unparse(test.left)),
                                      _ast.Constant(value=_ast.unparse(test.comparators[0]))],
                                keywords=[]
                            ))
                        ]
                return [_ast.Expr(value=_ast.Call(
                    func=_ast.Name(id="_check_expr", ctx=_ast.Load()),
                    args=[test, _ast.Constant(value=_ast.unparse(test))],
                    keywords=[]
                ))]

        if not any(isinstance(t, _AssertRewriter) for t in _ip.ast_transformers):
            _ip.ast_transformers.append(_AssertRewriter())
except NameError:
    pass

### 5.2.1 · Array-backed max-heap skeleton  `Core`

**Task.** Create `MaxHeap` storing nodes in a list `data`. Add `root()`, `last()`, `__len__`, and static helpers `left_child(i)`, `right_child(i)`, `parent(i)`.

**Example.** `left_child(0)` -> `1`, `right_child(0)` -> `2`, `parent(4)` -> `1`

**Big O.** All O(1). Storing the heap in an array solves "find the last node" for free: it is the last element.

**Note.** For index i: left child 2i + 1, right child 2i + 2, parent (i - 1) // 2.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-1) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-1)

In [ ]:
# 5.2.1 Array-backed max-heap skeleton
class MaxHeap:
    def __init__(self):
        raise NotImplementedError

    def __len__(self):
        raise NotImplementedError

    def root(self):
        raise NotImplementedError

    def last(self):
        raise NotImplementedError

    @staticmethod
    def left_child(i):
        raise NotImplementedError

    @staticmethod
    def right_child(i):
        raise NotImplementedError

    @staticmethod
    def parent(i):
        raise NotImplementedError

In [ ]:
def test_ex_5_2_1():
    assert MaxHeap.left_child(0) == 1
    assert MaxHeap.right_child(0) == 2
    assert MaxHeap.parent(4) == 1
    h = MaxHeap()
    assert len(h) == 0
    h.data = [9, 5, 3]
    assert h.root() == 9
    assert h.last() == 3
    assert len(h) == 3


test_ex_5_2_1()
print('5.2.1 passed')

### 5.2.2 · Insert and trickle up  `Core`

**Task.** Add `insert(value)`: append it as the new last node, then swap it with its parent while it is larger than the parent.

**Example.** insert 3, 9, 5, 12 -> `root()` is `12`

**Big O.** O(log N): at most one swap per level of the tree.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-2) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-2)

In [ ]:
# 5.2.2 Insert and trickle up
def insert(self, value):
    raise NotImplementedError


MaxHeap.insert = insert

In [ ]:
def test_ex_5_2_2():
    h = MaxHeap()
    for v in (3, 9, 5, 12):
        h.insert(v)
    assert h.root() == 12
    assert len(h) == 4
    assert all((h.data[MaxHeap.parent(i)] >= h.data[i] for i in range(1, len(h.data))))


test_ex_5_2_2()
print('5.2.2 passed')

### 5.2.3 · Delete the root and trickle down  `Core`

**Task.** Add `delete()` returning the root: move the last node into the root, then swap it with its larger child while a child is larger.

**Example.** After inserting 3, 9, 5, 12 -> `delete()` returns 12, 9, 5, 3 in that order

**Big O.** O(log N).

**Note.** Always swap with the larger child; swapping with the smaller one would put a smaller value above a larger one.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-3) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-3)

In [ ]:
# 5.2.3 Delete the root and trickle down
def delete(self):
    raise NotImplementedError


MaxHeap.delete = delete

In [ ]:
def test_ex_5_2_3():
    h = MaxHeap()
    for v in (3, 9, 5, 12):
        h.insert(v)
    assert [h.delete() for _ in range(5)] == [12, 9, 5, 3, None]


test_ex_5_2_3()
print('5.2.3 passed')

### 5.2.4 · Heap sort with your heap  `Practice`

**Task.** Write `heap_sort(values)`: insert everything into a `MaxHeap`, delete repeatedly, reverse.

**Example.** `[5, 1, 9, 3, 7]` -> `[1, 3, 5, 7, 9]`

**Big O.** O(N log N) time, O(N) space as written.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-4) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-4)

In [ ]:
# 5.2.4 Heap sort with your heap
def heap_sort(values):
    raise NotImplementedError

In [ ]:
def test_ex_5_2_4():
    rng = random.Random(1)
    data = [rng.randint(-100, 100) for _ in range(200)]
    assert heap_sort(data) == sorted(data)
    assert heap_sort([]) == []


test_ex_5_2_4()
print('5.2.4 passed')

### 5.2.5 · Jobs by deadline  `Core`

**Task.** Build `DeadlineQueue` with `add(job, deadline)` and `next_job()`, serving the earliest deadline first and, on a tie, whichever job was added first; `next_job()` returns `None` when empty. Use `heapq` (a min-heap) by pushing `(deadline, arrival_number, job)`.

**Example.** add ('report', 5), ('backup', 2), ('email', 2) -> next is 'backup', then 'email', then 'report'

**Big O.** O(log N) per add and per next. A sorted list would be O(N) to add.

**Note.** A heap is not stable: without the arrival number, two jobs with the same deadline could come out in either order.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-5) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-5)

In [ ]:
# 5.2.5 Jobs by deadline
class DeadlineQueue:
    def __init__(self):
        raise NotImplementedError

    def add(self, job, deadline):
        raise NotImplementedError

    def next_job(self):
        raise NotImplementedError

In [ ]:
def test_ex_5_2_5():
    q = DeadlineQueue()
    q.add('report', 5)
    q.add('backup', 2)
    q.add('email', 2)
    assert [q.next_job() for _ in range(4)] == ['backup', 'email', 'report', None]
    q.add('late', 9)
    q.add('soon', 1)
    assert q.next_job() == 'soon'


test_ex_5_2_5()
print('5.2.5 passed')

### 5.2.6 · Top K largest  `Practice`

**Task.** Write `top_k(nums, k)` keeping a min-heap of size k: push each number, pop when the heap grows past k. Return the k largest, largest first.

**Example.** `[3, 1, 9, 7, 5, 8]`, k = 3 -> `[9, 8, 7]`

**Big O.** O(N log K) time, O(K) space. Better than sorting when K is much smaller than N.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-6) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-6)

In [ ]:
# 5.2.6 Top K largest
def top_k(nums, k):
    raise NotImplementedError

In [ ]:
def test_ex_5_2_6():
    assert top_k([3, 1, 9, 7, 5, 8], 3) == [9, 8, 7]


test_ex_5_2_6()
print('5.2.6 passed')

### 5.2.7 · Is this array a max-heap?  `Core`

**Task.** Write `is_max_heap(arr)` checking that every parent is at least as large as its children, using the index formulas.

**Example.** `[100, 88, 25, 87, 16]` -> `True`; `[10, 20]` -> `False`

**Big O.** O(N).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-7) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-7)

In [ ]:
# 5.2.7 Is this array a max-heap?
def is_max_heap(arr):
    raise NotImplementedError

In [ ]:
def test_ex_5_2_7():
    assert is_max_heap([100, 88, 25, 87, 16])
    assert not is_max_heap([10, 20])
    assert is_max_heap([])


test_ex_5_2_7()
print('5.2.7 passed')

### 5.2.8 · Merge K sorted lists  `Stretch`

**Task.** Write `merge_k_sorted(lists)` using a heap holding one `(value, list_index, element_index)` entry per list.

**Example.** `[[1, 4, 7], [2, 5], [0, 9]]` -> `[0, 1, 2, 4, 5, 7, 9]`

**Big O.** O(N log K) for N total values across K lists.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-8) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-8)

In [ ]:
# 5.2.8 Merge K sorted lists
def merge_k_sorted(lists):
    raise NotImplementedError

In [ ]:
def test_ex_5_2_8():
    assert merge_k_sorted([[1, 4, 7], [2, 5], [0, 9], []]) == [0, 1, 2, 4, 5, 7, 9]
    assert merge_k_sorted([]) == []


test_ex_5_2_8()
print('5.2.8 passed')

### 5.2.9 · Running median  `Practice`

**Task.** Build `RunningMedian` with `add(x)` and `median()` (a float; at least one value has been added). Keep the lower half in a max-heap (store negatives with `heapq`) and the upper half in a min-heap, rebalancing so the lower half has the same size or one more.

**Example.** add 5 -> `5.0`; add 15 -> `10.0`; add 1 -> `5.0`; add 3 -> `4.0`

**Big O.** O(log N) per `add`, O(1) per `median`, O(N) space: a max-heap holds the lower half and a min-heap the upper half, and their tops straddle the median.

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-9) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-9)

In [ ]:
# 5.2.9 Running median
class RunningMedian:
    def __init__(self):
        raise NotImplementedError

    def add(self, x):
        raise NotImplementedError

    def median(self):
        raise NotImplementedError

In [ ]:
def test_ex_5_2_9():
    m = RunningMedian()
    got = []
    for x in (5, 15, 1, 3):
        m.add(x)
        got.append(m.median())
    assert got == [5.0, 10.0, 5.0, 4.0]
    stream, seen, m = ([8, -2, 7, 7, 0, 11, 3, 3, -5, 20], [], RunningMedian())
    for x in stream:
        m.add(x)
        seen.append(x)
        s, n = (sorted(seen), len(seen))
        assert m.median() == (s[n // 2] if n % 2 else (s[n // 2 - 1] + s[n // 2]) / 2)


test_ex_5_2_9()
print('5.2.9 passed')

### 5.2.10 · Sort a nearly sorted array with a heap  `Practice`

**Task.** Every value in `nums` is at most `k` positions from where it belongs in sorted order. Write `sort_nearly_sorted(nums, k)` that returns a new sorted list in O(N log K): keep a min-heap of the next `k + 1` values and repeatedly move its smallest to the output.

**Example.** `[3, 1, 2, 6, 4, 5, 9, 7, 8], 2` -> `[1, 2, 3, 4, 5, 6, 7, 8, 9]`

**Big O.** O(N log K) time, O(K) space: the smallest remaining value is always among the next K + 1, so a heap of that size can hand it out.

**Note.** Compare 3.1.16, which handles the same input with insertion sort in O(N * K).

[Workbook](https://bloomcode.github.io/dsa/python/workbook.html#ex-5-2-10) · [Solution in the answer key](https://bloomcode.github.io/dsa/python/answer-key.html#ex-5-2-10)

In [ ]:
# 5.2.10 Sort a nearly sorted array with a heap
def sort_nearly_sorted(nums, k):
    raise NotImplementedError

In [ ]:
def test_ex_5_2_10():
    assert sort_nearly_sorted([3, 1, 2, 6, 4, 5, 9, 7, 8], 2) == list(range(1, 10))
    assert sort_nearly_sorted([], 3) == []
    assert sort_nearly_sorted([2, 1, 4, 3], 1) == [1, 2, 3, 4]
    assert sort_nearly_sorted([1, 2, 3], 0) == [1, 2, 3]
    assert sort_nearly_sorted([5, 4, 3, 2, 1], 4) == [1, 2, 3, 4, 5]
    assert sort_nearly_sorted([2, 2, 1], 2) == [1, 2, 2]


test_ex_5_2_10()
print('5.2.10 passed')